In [3]:
import pandas as pd
import numpy as np

In [10]:
kb_df=pd.read_csv(r"C:\Users\VAIBHAV\Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv")
kb_df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [11]:
kb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   flags        26872 non-null  object
 1   instruction  26872 non-null  object
 2   category     26872 non-null  object
 3   intent       26872 non-null  object
 4   response     26872 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB


In [12]:
kb_df.describe()

,flags,instruction,category,intent,response
count,26872,26872,26872,26872,26872
unique,394,24635,11,27,26870
top,BL,is it possible to place an order from {{Delive...,ACCOUNT,contact_customer_service,"Firstly, I truly understand how pivotal the {{..."
freq,5212,8,5986,1000,2


In [16]:
print("Shape:",kb_df.shape)
print("Nulls:",kb_df.isnull().sum())
print(kb_df['category'].value_counts())
print(kb_df['intent'].value_counts().head(20))

Shape: (26872, 5)
Nulls: flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64
category
ACCOUNT         5986
ORDER           3988
REFUND          2992
CONTACT         1999
INVOICE         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64
intent
contact_customer_service    1000
complaint                   1000
check_invoice               1000
switch_account              1000
edit_account                1000
contact_human_agent          999
check_payment_methods        999
delivery_period              999
newsletter_subscription      999
get_invoice                  999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
set_up_shipping_address      997
check_refund_policy          997
create_acc

In [18]:
import json
import re

In [19]:
kb_df['clean_instruction'] = kb_df['instruction'].str.lower()

kb_df['clean_instruction'] = kb_df['clean_instruction'].apply(
    lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x)
)

In [20]:
unwanted_terms = [
    'order number',
    'ordernumber'
]

for term in unwanted_terms:
    kb_df['clean_instruction'] = kb_df['clean_instruction'].str.replace(term, '', regex=False)

In [21]:
def retrieve_response(user_query):

    user_query = user_query.lower()

    for index, row in kb_df.iterrows():

        if any(word in row['clean_instruction']
               for word in user_query.split()):

            return {
                'intent': row['intent'],
                'response': row['response']
            }

    return "No matching support solution found."

In [22]:
retrieve_response(
    "I want to cancel my order"
)

{'intent': 'cancel_order',
 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}

In [23]:
def retrieve_response(user_query):

    user_query = user_query.lower()

    for index, row in kb_df.iterrows():

        if any(word in row['clean_instruction']
               for word in user_query.split()):

            return {
                'category': row['category'],
                'intent': row['intent'],
                'response': row['response']
            }

    return {
        'category': 'Unknown',
        'intent': 'Unknown',
        'response': 'No matching support solution found.'
    }

In [24]:
escalation_keywords = [
    'angry',
    'refund',
    'cancel',
    'frustrated',
    'not working'
]

In [25]:
def check_escalation(user_query):

    user_query = user_query.lower()

    for word in escalation_keywords:

        if word in user_query:
            return True

    return False

In [26]:
query = "I am frustrated because my payment failed"

result = retrieve_response(query)

print(result)

if check_escalation(query):
    print("Escalation Recommended")

{'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}
Escalation Recommended


In [27]:
def support_assistant(user_query):
    retrieval_result = retrieve_response(user_query)
    escalation_required = check_escalation(user_query)

    return {
        'category': retrieval_result['category'],
        'intent': retrieval_result['intent'],
        'response': retrieval_result['response'],
        'escalation_required': escalation_required
    }

In [28]:
support_assistant("I am frustrated because my payment failed")

{'category': 'ORDER',
 'intent': 'cancel_order',
 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.",
 'escalation_required': True}

## Assistant Workflow V1

This section combines KB retrieval and escalation logic into a single support assistant function. The assistant identifies a relevant support response from the knowledge base and flags cases that may require human escalation.